# Bulk fetch of Mapillary map_features

Notebook interface for running `src/fetch_mapillary_features.py`.

**What this does:**
- For every feature's `StreetImageLink` (bbox) in both countries' GeoJSON, fetch the 27 VRU-related categories from the Mapillary `/map_features` API.
- Results are saved to `data/mapillary/map_features_{country}.json` (OBJECTID -> list of features).
- Checkpointing (`.jsonl`) allows interrupting and resuming partway through.

**Prerequisite:** `MAPILLARY_TOKEN` must be set in `.env`.

In [1]:
import os
import sys

if os.path.basename(os.getcwd) == "notebooks":
 os.chdir("..")
sys.path.insert(0, "src")
print("cwd:", os.getcwd)

cwd: /Users/kentaroiio/Documents/github/adb-ai-for-safer-roads-safer-speeds-challenge-dev


## 1. Check the token

In [2]:
from dotenv import load_dotenv

load_dotenv
token = os.environ.get("MAPILLARY_TOKEN", "")
if not token:
 raise EnvironmentError("MAPILLARY_TOKEN is not set in.env")
print(f"MAPILLARY_TOKEN: {token[:6]}... (length {len(token)})")

MAPILLARY_TOKEN: MLY|37... (length 54)


## 2. Dry-run (no API calls)

Just displays the tasks for the first 5 features. Lets you check the bbox and sub-tile split count.

In [3]:
from fetch_mapillary_features import DATASETS, run

await run(DATASETS[0], token=token, concurrency=15, limit=5, dry_run=True)

23:29:32 INFO [maharashtra] Loading data/raw/ADB_Innovation_Maharashtra.geojson
23:29:32 INFO [maharashtra] Checkpoint: 0 already done
23:29:32 INFO [maharashtra] Pending: 5 features
23:29:32 INFO [maharashtra] --dry-run: printing first 5 tasks


 OBJECTID=1 link=73.5544591,16.2774648,73.53445,16.2401913 subtiles=1
 OBJECTID=2 link=73.9651122,21.0463691,73.9910297,21.0647687 subtiles=1
 OBJECTID=3 link=74.9005386,19.0351416,74.8847951,19.020001 subtiles=1
 OBJECTID=4 link=76.593981,19.1964024,76.6112397,19.1893524 subtiles=1
 OBJECTID=5 link=76.0436565,17.370428,76.0713028,17.3902438 subtiles=1


## 3. Smoke test (uses the API, 10 items per country)

Verify API connectivity and the output format before the full run.

> This writes to the checkpoint, so run the deletion cell in Section 4 before the full run.

In [4]:
await run(DATASETS[0], token=token, concurrency=15, limit=10, dry_run=False)

23:29:32 INFO [maharashtra] Loading data/raw/ADB_Innovation_Maharashtra.geojson
23:29:33 INFO [maharashtra] Checkpoint: 0 already done
23:29:33 INFO [maharashtra] Pending: 10 features


maharashtra: 0%| | 0/10 [00:00<?, ?seg/s]

23:30:03 INFO [maharashtra] 10 / 10 done (found 0 features so far)
23:30:03 INFO [maharashtra] Writing data/mapillary/map_features_maharashtra.json (10 entries)
23:30:03 INFO [maharashtra] Done.


In [5]:
import json

with open("data/mapillary/map_features_maharashtra.json") as f:
 sample = json.load(f)

print(f"Entry count: {len(sample)}")
for oid, feats in list(sample.items)[:5]:
 print(f" OBJECTID={oid}: {len(feats)} features")
 for feat in feats[:2]:
 print(f" {feat['object_value']} @ {feat['geometry']['coordinates']}")

Entry count: 10
 OBJECTID=2: 0 features
 OBJECTID=5: 0 features
 OBJECTID=6: 0 features
 OBJECTID=9: 0 features
 OBJECTID=1: 0 features


## 4. Full run

- Maharashtra: 14,082 features / Thailand: 55,884 features
- Total requests: ~74,842
- Estimated duration: 15-45 minutes (15 concurrent)

If interrupted, the checkpoint remains, so simply re-running the cell resumes from where it left off.

In [6]:
from pathlib import Path

# Only run this to clear smoke-test leftovers before a full run.
# Skip this cell if resuming.
for country in ["maharashtra", "thailand"]:
 for p in [
 Path(f"data/mapillary/.map_features_{country}.jsonl"),
 Path(f"data/mapillary/map_features_{country}.json"),
 ]:
 if p.exists:
 p.unlink
 print(f"Deleted: {p}")

Deleted: data/mapillary/.map_features_maharashtra.jsonl
Deleted: data/mapillary/map_features_maharashtra.json


In [7]:
# Maharashtra full fetch
await run(DATASETS[0], token=token, concurrency=15)

23:30:03 INFO [maharashtra] Loading data/raw/ADB_Innovation_Maharashtra.geojson
23:30:04 INFO [maharashtra] Checkpoint: 0 already done
23:30:04 INFO [maharashtra] Pending: 14082 features


maharashtra: 0%| | 0/14082 [00:00<?, ?seg/s]

23:34:56 INFO [maharashtra] 500 / 14082 done (found 16 features so far)
23:42:30 INFO [maharashtra] 1000 / 14082 done (found 25 features so far)
23:48:12 INFO [maharashtra] 1500 / 14082 done (found 73 features so far)
23:53:34 INFO [maharashtra] 2000 / 14082 done (found 129 features so far)
23:58:28 INFO [maharashtra] 2500 / 14082 done (found 183 features so far)
00:05:56 INFO [maharashtra] 3000 / 14082 done (found 209 features so far)
00:13:33 INFO [maharashtra] 3500 / 14082 done (found 229 features so far)
00:17:56 INFO [maharashtra] 4000 / 14082 done (found 250 features so far)
00:22:34 INFO [maharashtra] 4500 / 14082 done (found 269 features so far)
00:28:28 INFO [maharashtra] 5000 / 14082 done (found 308 features so far)
00:32:48 INFO [maharashtra] 5500 / 14082 done (found 319 features so far)
00:37:09 INFO [maharashtra] 6000 / 14082 done (found 365 features so far)
00:41:35 INFO [maharashtra] 6500 / 14082 done (found 395 features so far)
00:44:49 INFO [maharashtra] 7000 / 14082 d

In [8]:
# Thailand full fetch
await run(DATASETS[1], token=token, concurrency=15)

01:40:35 INFO [thailand] Loading data/raw/ADB_Innovation_Thailand.geojson
01:40:36 INFO [thailand] Checkpoint: 0 already done
01:40:36 INFO [thailand] Pending: 55884 features


thailand: 0%| | 0/55884 [00:00<?, ?seg/s]

01:41:46 INFO [thailand] 500 / 55884 done (found 1918 features so far)
01:43:45 INFO [thailand] 1000 / 55884 done (found 4360 features so far)
01:45:24 INFO [thailand] 1500 / 55884 done (found 7953 features so far)
01:46:43 INFO [thailand] 2000 / 55884 done (found 13055 features so far)
01:48:40 INFO [thailand] 2500 / 55884 done (found 16931 features so far)
01:50:20 INFO [thailand] 3000 / 55884 done (found 20469 features so far)
01:51:42 INFO [thailand] 3500 / 55884 done (found 23641 features so far)
01:52:47 INFO [thailand] 4000 / 55884 done (found 25405 features so far)
01:53:44 INFO [thailand] 4500 / 55884 done (found 26773 features so far)
01:54:46 INFO [thailand] 5000 / 55884 done (found 27432 features so far)
01:56:48 INFO [thailand] 5500 / 55884 done (found 29822 features so far)
01:57:41 INFO [thailand] 6000 / 55884 done (found 30573 features so far)
02:00:59 INFO [thailand] 6500 / 55884 done (found 31823 features so far)
02:02:24 INFO [thailand] 7000 / 55884 done (found 43904

## 5. Results summary

In [9]:
from collections import Counter

for country in ["maharashtra", "thailand"]:
 path = Path(f"data/mapillary/map_features_{country}.json")
 if not path.exists:
 print(f"{country}: file not found")
 continue

 with open(path) as f:
 data = json.load(f)

 total_feats = sum(len(v) for v in data.values)
 non_empty = sum(1 for v in data.values if v)
 cat_counter: Counter = Counter
 for feats in data.values:
 for feat in feats:
 cat_counter[feat["object_value"]] += 1

 print(f"\n=== {country.upper} ===")
 print(f" Road segments: {len(data)}")
 print(f" With at least 1 map_feature: {non_empty} ({non_empty / len(data) * 100:.1f}%)")
 print(f" Total map_features: {total_feats}")
 print(" Counts by category (top 10):")
 for cat, cnt in cat_counter.most_common(10):
 print(f" {cnt:6d} {cat}")


=== MAHARASHTRA ===
 Road segments: 14082
 With at least 1 map_feature: 517 (3.7%)
 Total map_features: 3026
 Counts by category (top 10):
 1322 marking--discrete--crosswalk-zebra
 1076 construction--flat--driveway
 525 warning--pedestrians-crossing--g1
 59 warning--pedestrians-crossing--g5
 23 marking--discrete--symbol--bicycle
 7 construction--flat--crosswalk-plain
 6 object--bike-rack
 4 warning--pedestrians-crossing--g4
 2 regulatory--shared-path-bicycles-and-pedestrians--g1
 2 warning--pedestrians-crossing--g12

=== THAILAND ===
 Road segments: 55884
 With at least 1 map_feature: 8206 (14.7%)
 Total map_features: 168953
 Counts by category (top 10):
 133947 construction--flat--driveway
 24667 marking--discrete--crosswalk-zebra
 5228 warning--pedestrians-crossing--g4
 1899 marking--discrete--symbol--bicycle
 1098 object--bike-rack
 938 construction--flat--crosswalk-plain
 751 warning--school-zone--g2
 190 regulatory--shared-path-bicycles-and-pedestrians--g1
 63 information--hospit